# cli

> Ramabana in a terminal: a transcript of blocks, a status bar, and one line to type in.

Built on [teleprint](https://github.com/answerdotai/teleprint), whose centre is the same as
this app's: an append-mostly transcript whose durable rendering is the terminal's own
scrollback. Tool calls are *blocks* rather than lines, which is what makes them foldable --
the answer stays readable and the thirty tool results it took are one click away.

In [ ]:
#| default_exp cli

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| hide
import asyncio, tempfile, threading
from fastcore.test import test_eq, test_fail
from teleprint.keys import Key
from teleprint.testing import EmuTty
from ramabana.testing import FullHost, fake_agent

In [ ]:
#| export
import asyncio, sys, threading
from pathlib import Path
from rich.text import Text
from fastcore.script import call_parse
from teleprint.buffer import Buffer
from teleprint.compositor import Compositor
from teleprint.tty import RealTty
from ramabana.core import agent_err
from ramabana.tools import WRITE_TOOLS, LocalHost
from ramabana.agent import Agent, Approvals, answer_md

## Blocks and keys

Six kinds of block, one gutter each. The gutter glyph is also the click target, so folding
a tool result is a click on its dot -- which is the entire reason a tool result is a block
rather than part of the reply.

In [ ]:
#| export
GUTTERS = {'user':  (Text('› ', style='bold cyan'), Text('  ')),
           'reply': (Text('  '),                    Text('  ')),
           'tool':  (Text('· ', style='dim'),        Text('  ')),
           'ask':   (Text('? ', style='bold yellow'), Text('  ')),
           'note':  (Text('! ', style='yellow'),     Text('  ')),
           'error': (Text('✗ ', style='bold red'),   Text('  '))}

FOLD = 12          # a block taller than this is born folded; click or ctrl-O to open it
HELP = """keys    enter send · ctrl+o fold the last block · ctrl+c stop the turn · ctrl+d quit
approve y approve · n refuse · a approve all · or type a reason and press enter to refuse
extra   /help, and every ramabana command: /model /cost /compact /skills /tools /reload"""

In [ ]:
print(HELP)

keys    enter send · ctrl+o fold the last block · ctrl+c stop the turn · ctrl+d quit
approve y approve · n refuse · a approve all · or type a reason and press enter to refuse
extra   /help, and every ramabana command: /model /cost /compact /skills /tools /reload


## The turn

`agent.stream` is a blocking generator on the model's thread, so its chunks come back over a
queue rather than being awaited. The loop has to stay free the entire time: an approval that
cannot be answered until the turn ends is not an approval, and a tool call that cannot
repaint until then is not a live feed.

In [ ]:
#| export
async def run_turn(ui, prompt):
    """One turn, streamed into the transcript.

    The agent's `stream` is a blocking generator on the model's own thread, so the chunks
    come back over a queue rather than being awaited: the loop has to stay free the whole
    time, or an approval could never be answered and a tool call could never repaint.
    """
    loop, q = asyncio.get_running_loop(), asyncio.Queue()
    def pump():
        try:
            for chunk in ui.agent.stream(prompt): loop.call_soon_threadsafe(q.put_nowait, chunk)
        except Exception as e: loop.call_soon_threadsafe(q.put_nowait, agent_err(e))
        finally: loop.call_soon_threadsafe(q.put_nowait, None)
    threading.Thread(target=pump, daemon=True).start()
    blk = None
    try:
        while (chunk := await q.get()) is not None: blk = ui.stream(blk, chunk)
    finally:
        ui.turn = None
        for p in ui.agent.problems: ui.say(Text(p), 'error')
        ui.agent.clear_problems()
        ui.paint()
    return blk

## The surface

`Ui` is the whole terminal surface, and every method on it is synchronous and free of tty
work -- which is what lets the tests below drive it against an emulated terminal instead of
mocking one. The one hazard it has to handle is thread affinity: activity and approval
callbacks arrive on the model's worker thread, and a compositor may only be touched from the
loop thread, so everything they do goes through `_post`.

It carries three responsibilities: painting the tail (status bar plus input line), turning
tool calls and approval requests into blocks, and deciding what one keystroke means -- which
is either a coroutine for the caller to spawn, `'quit'`, or nothing.

In [ ]:
#| export
class Ui:
    """The terminal surface: a transcript of blocks, a status bar, and one line to type in.

    Every method here is synchronous and free of tty work, so the whole surface can be
    driven in a test against an emulated terminal -- which is why the async loop below is
    as small as it is.

    Callbacks arrive from the model's worker thread (`Activity.on_change`, `Approvals`), and
    a compositor may only be touched from the loop thread, so everything they do goes
    through `_post`. Without a loop registered it calls straight through, which is what
    makes the synchronous tests possible.
    """

    def __init__(self, comp, agent, loop=None):
        self.comp, self.agent, self.loop = comp, agent, loop
        self.buf = Buffer()
        self.ask = None            # the `Ask` waiting on an answer, or None
        self.turn = None           # the running turn's task, or None
        self.acts = {}             # act id -> its block
        self.hint = ''
        self._reply = ''           # the reply text so far, so a streamed block repaints whole
        agent.activity.on_change = self.on_act
        if agent.approvals is not None: agent.approvals.listen(self.on_ask, self.on_answer)

    def _post(self, fn, *a):
        "Run `fn` on the loop thread, or now when there is no loop (a test, or startup)."
        if self.loop is None: return fn(*a)
        self.loop.call_soon_threadsafe(fn, *a)

    # -- the tail ------------------------------------------------------------
    def status(self):
        "The status bar: what is loaded, how full it is, and what it has cost so far."
        s = self.agent.status()
        state = 'working' if s['busy'] else ('ready' if s['ready'] else 'idle')
        bits = [f"{s['model']}", state, f"{s['ntools']} tools", f"{s['nskills']} skills",
                f"{round(s['pct_full'] * 100)}% ctx"]
        if s['compactions']: bits.append(f"{s['compactions']} compaction(s)")
        if self.agent.use.total: bits.append(s['usage'])
        if s['problems']: bits.append(f"{len(s['problems'])} problem(s)")
        return Text(' ' + ' · '.join(bits), style='reverse')

    #: The approval prompt. `y`/`n`/`a` answer only while nothing is typed, because a reason
    #: like "put it in docs/ instead" contains all three letters -- so once there is text, the
    #: letters are text and `enter` is the answer.
    ASKING = 'approve? [y/n/a, or a reason + enter] '

    def prompt(self):
        "The input line: an approval question when one is pending, otherwise the prompt."
        if self.ask is not None:
            return Text(self.ASKING, style='bold yellow') + Text(self.buf.text)
        return Text('› ', style='bold cyan') + Text(self.buf.text)

    def paint(self):
        pre = self.ASKING if self.ask is not None else '› '
        tail = [self.status()] + ([Text(self.hint, style='dim')] if self.hint else []) + [self.prompt()]
        self.comp.set_tail(*tail, cursor=(len(tail) - 1, self.buf.cell_cursor(pre)))

    # -- the transcript ------------------------------------------------------
    def say(self, body, kind='reply', fold=FOLD):
        "Print one block. Text is wrapped in `Text` so nothing in a model reply is read as markup."
        return self.comp.print_block(body if isinstance(body, Text) else Text(str(body)),
                                     gutter=GUTTERS.get(kind, GUTTERS['reply']),
                                     tag=kind, collapse_at=fold)

    def on_act(self, act):
        "Called twice per tool call, from the model's thread: once running, once finished."
        self._post(self._act, act)

    def _act(self, act):
        line = Text(act.line())
        blk = self.acts.get(act.id)
        if blk is None:
            self.acts[act.id] = self.say(line, 'tool')
            return self.paint()
        body = [line] if not act.detail else [line, Text(act.detail, style='dim')]
        self.comp.set_body(blk, *body)
        self.comp.refresh_block(blk)
        self.paint()

    # -- approvals -----------------------------------------------------------
    def on_ask(self, ask):
        "A write is waiting on a person. Print what it would do, and take over the input line."
        self._post(self._ask, ask)

    def _ask(self, ask):
        self.ask = ask
        self.buf.clear()
        self.say(Text(ask.summary, style='bold') + Text('\n\n') + Text(ask.preview), 'ask', fold=None)
        self.paint()

    def on_answer(self, ask):
        self._post(self._answered, ask)

    def _answered(self, ask):
        if self.ask is not None and self.ask.id == ask.id: self.ask = None
        self.say(Text(answer_md(ask).replace('**', '')), 'note')
        self.paint()

    def answer(self, ok, session=False):
        """Answer the pending request, using whatever has been typed as the reason.

        A refusal with a reason is the point of the gate: it reaches the model, which can
        change approach instead of retrying the same edit.
        """
        if self.ask is None: return None
        note, self.buf.text = self.buf.text.strip(), ''
        return self.agent.approvals.answer(self.ask.id, ok, note, session=session)

    # -- input ---------------------------------------------------------------
    def submit(self):
        """Handle the typed line. Returns a coroutine for a turn, or None when it was handled here.

        Slash commands are answered by the agent, so every command the IDE has works here
        too -- there is one implementation of `/model`, and it is not in a frontend.
        """
        line = self.buf.text.strip()
        self.buf.clear()
        if not line: return None
        self.say(Text(line), 'user')
        if line in ('/help', '/?'):
            self.say(Text(HELP), 'note', fold=None)
            return None
        if line.startswith('/'):
            out = self.agent.command(line)
            self.say(Text(out) if out is not None else Text(f'unknown command: {line}'),
                     'note' if out is not None else 'error', fold=None)
            return None
        return run_turn(self, line)

    def on_key(self, k):
        "One keystroke. Returns a coroutine to spawn, `'quit'`, or None."
        if self.ask is not None:
            bare = not self.buf.text.strip()
            if bare and k.name in ('y', 'Y'):     self.answer(True)
            elif bare and k.name in ('n', 'N'):   self.answer(False)
            elif bare and k.name in ('a', 'A'):   self.answer(True, session=True)
            elif k.name == 'enter':               self.answer(False)   # a typed reason is a refusal
            elif k.name == 'ctrl+y':              self.answer(True)    # ...unless approved with it as guidance
            elif k.name == 'ctrl+c':              self.answer(False)     # stopping the turn refuses what it was waiting on
            else: self.buf.handle(k)
            return self.paint()
        if k.name == 'ctrl+d' and not self.buf.text: return 'quit'
        if k.name == 'ctrl+c':
            if self.turn is not None:
                self.agent.cancel()
                self.turn.cancel()
                self.turn = None
                self.say(Text('stopped'), 'note')
            self.buf.clear()
            return self.paint()
        if k.name == 'enter':
            coro = self.submit()
            self.paint()
            return coro
        if k.name == 'ctrl+o':
            live = [b for b in self.comp.blocks.values() if not b.committed and b.height > 1]
            if live: self.comp.toggle(live[-1])
            return self.paint()
        self.buf.handle(k)
        self.paint()

    def stream(self, blk, chunk):
        """Grow the reply as the model produces it, repainting the block from its whole text.

        Not `Compositor.extend`, for two reasons: each extend appends at least one row, so a
        token stream would print one word per line; and a tool call that starts mid-reply
        makes the reply no longer the last block, which extend refuses. Repainting from the
        accumulated text is correct under both, and a reply is small enough to repaint.
        """
        self._reply = (self._reply + chunk) if blk is not None else chunk
        if blk is None: return self.say(Text(self._reply), 'reply', fold=None)
        self.comp.set_body(blk, Text(self._reply))
        self.comp.refresh_block(blk)
        return blk

A headless terminal, a real compositor, and an agent whose model is a fake: the whole
surface, in a notebook.

In [ ]:
tty = EmuTty(72, 14)
comp = await Compositor(tty).start()
agent, be = fake_agent(replies=['`threshold` is in `ramabana/runtime.py`, and it caps the reserve.'])
ui = Ui(comp, agent)
comp.on_key = ui.on_key          # a coroutine returned by a handler is spawned by the dispatcher
ui.paint()
print(tty.term.text())

 ornith-9b · idle · 14 tools · 15 skills · 0% ctx
›


That is the resting state -- the status bar says what is loaded and how full it is, and the
cursor sits on the prompt. Everything above the last two rows is the transcript.

In [ ]:
test_eq(tty.term.text().splitlines()[-1], '›')
[l for l in tty.term.text().splitlines() if 'tools' in l]

[' ornith-9b · idle · 14 tools · 15 skills · 0% ctx']

Typing goes through the real key parser: these are the bytes a terminal sends, not
synthesised `Key` objects.

In [ ]:
comp.on_bytes(b'where is the threshold?')
tty.term.text().splitlines()[-1]

'› where is the threshold?'

Enter prints the question as a block and spawns the turn. Awaiting a moment lets the model
thread run -- in the real app that wait is the event loop, which is why nothing here blocks
it.

In [ ]:
comp.on_bytes(b'\r')
await asyncio.sleep(0.4)
print(tty.term.text())

› where is the threshold?
· 🔍 Search where is the threshold?
  no matches (memory)
  `threshold` is in `ramabana/runtime.py`, and it caps the reserve.
 ornith-9b · ready · 14 tools · 15 skills · 2% ctx · 15 tok · in 10 ·
out 5 · model
›


The transcript now holds the question, the tool calls the agent made on the way, and the
reply -- each one a block with its own gutter.

In [ ]:
[(b.tag, b.height) for b in comp.blocks.values()]

[('user', 1), ('tool', 2), ('reply', 1)]

In [ ]:
test_eq(agent.calls[0][0], 'search_code')       # the preflight ran, and the feed recorded it
test_eq(any('threshold' in str(b.body) for b in comp.blocks.values()), True)
be.sent and len(be.sent)

1

## Folding

A block taller than `FOLD` is born folded, so a four-hundred-line file view costs one row
until it is asked for. Clicking its gutter toggles it; so does ctrl-O on the last block.

In [ ]:
long = ui.say(Text('\n'.join(f'line {i}' for i in range(40))), 'tool')
long.collapsed, long.height

(True, 40)

In [ ]:
comp.toggle(long)
test_eq(long.collapsed, False)
comp.toggle(long)
long.collapsed

True

A one-line block has nothing to hide, and toggling it is a no-op rather than an error.

In [ ]:
short = ui.say(Text('one line'), 'tool')
comp.toggle(short), short.collapsed

(None, False)

## Approvals in a terminal

The gate blocks the model's worker thread until a person answers, so the request arrives on
that thread and the answer comes from the loop. The input line becomes the answer: type a
reason, then `y`, `n`, or `a` for the rest of the session.

A write, asked for from another thread -- exactly as a tool call does it:

In [ ]:
tty2 = EmuTty(72, 12)
comp2 = await Compositor(tty2).start()
gated, _ = fake_agent(replies=['done'])
gated.approvals = Approvals(tools=WRITE_TOOLS, host=gated.host)
ui2 = Ui(comp2, gated)
comp2.on_key = ui2.on_key
ui2.paint()
threading.Thread(target=lambda: gated.approvals.request(
    'create_file', {'path': '/proj/notes.md', 'text': '# notes\n'}), daemon=True).start()
await asyncio.sleep(0.2)
print(tty2.term.text())

? create_file → /proj/notes.md

  /proj/notes.md  (new file, 8 chars)

  # notes

 ornith-9b · idle · 14 tools · 15 skills · 0% ctx
approve? [y/n/a, or a reason + enter]


The preview is what the person reads: the path, whether it overwrites, and the head of what
would be written. The prompt has become the approval question.

In [ ]:
test_eq(ui2.ask.tool, 'create_file')
tty2.term.text().splitlines()[-1]

'approve? [y/n/a, or a reason + enter]'

`y`, `n` and `a` answer only while nothing has been typed -- a reason like "put it in docs/
instead" contains all three letters, so once there is text the letters are text and `enter`
is the answer. Typing a reason and pressing it refuses *with* that reason, which is the whole
point of the gate: something the model can act on rather than the word "denied".

In [ ]:
comp2.on_bytes(b'put it in docs/ instead')
tty2.term.text().splitlines()[-1]

'approve? [y/n/a, or a reason + enter] put it in docs/ instead'

In [ ]:
comp2.on_bytes(b'\r')
await asyncio.sleep(0.2)
ui2.ask, gated.approvals.history[-1].reply()

(None, 'Denied by human operator. Reason given: put it in docs/ instead')

In [ ]:
test_eq(bool(gated.approvals.history[-1]), False)
test_eq(tty2.term.text().splitlines()[-1], '›')      # the prompt is back
[b.tag for b in comp2.blocks.values()][-2:]

['ask', 'note']

`a` is the deliberate bulk answer: it approves this request and switches the policy to
`auto` for the rest of the session, which is a thing a person does on purpose rather than by
holding down return.

In [ ]:
threading.Thread(target=lambda: gated.approvals.request('create_file', {'path': '/proj/b.md'}), daemon=True).start()
await asyncio.sleep(0.2)
comp2.on_bytes(b'a')
await asyncio.sleep(0.2)
gated.approvals.mode, gated.approvals.history[-1].note

('auto', 'approved for the rest of this session')

## Commands

Slash commands are answered by the agent, not by the terminal: there is one implementation of
`/model`, and it is not in a frontend. `/help` is the only one this layer owns, because the
keys it describes are this layer's.

In [ ]:
ui.buf.insert('/tools')
ui.submit()
[l for l in str(comp.blocks[max(comp.blocks)].body[0]).splitlines()[:4]]

['create_file', 'create_skill', 'delegate_parallel', 'delegate_search']

In [ ]:
ui.buf.insert('/nope')
test_eq(ui.submit(), None)
comp.blocks[max(comp.blocks)].tag

'error'

An empty line does nothing at all, rather than sending an empty turn to the model.

In [ ]:
test_eq(ui.submit(), None)
ui.buf.text

''

## Running it

`mk_agent` is the assembly: a `LocalHost` over the folders named on the command line, an
`Agent` over that, and the approval gate wired to both. `amain` is the tty loop, and it is
short because everything it could get wrong lives in `Ui`.

In [ ]:
#| export
def mk_agent(roots=('.',), model=None, approve='ask', web=True, **kw):
    "A `LocalHost` and an `Agent` over it, gated the way `approve` says."
    approvals = None if approve == 'none' else Approvals(tools=WRITE_TOOLS, mode=approve)
    host = LocalHost(roots, approvals=approvals, web=web)
    return Agent(host, model=model, approvals=approvals, **kw), host

In [ ]:
#| export
async def amain(agent, hint=''):
    "The tty loop: one terminal, one event loop, one place that owns the keyboard."
    tty = RealTty()
    tty.write('\x1b[?1000;1006h\x1b[?2004h')     # SGR mouse (clicks fold blocks) + bracketed paste
    done = asyncio.Event()
    try:
        comp = await Compositor(tty).start()
        ui = Ui(comp, agent, loop=asyncio.get_running_loop())
        ui.hint = hint
        comp.on_task_error = lambda e, t: ui.say(Text(f'{t.get_name()} failed: {e!r}'), 'error')
        def on_key(k):
            out = ui.on_key(k)
            if out == 'quit': return done.set()
            if out is not None:
                ui.turn = comp.spawn(out, name='turn')
                ui.paint()
        comp.on_key = on_key
        comp.on_paste = lambda text: (ui.buf.insert(text), ui.paint())
        comp.on_resize = lambda: (comp.resize(), ui.paint())
        comp.on_click = lambda blk: comp.toggle(blk)
        ui.say(Text(f'ramabana · {agent.note}\n{HELP}'), 'note', fold=None)
        ui.paint()
        loop = asyncio.get_running_loop()
        loop.add_reader(tty.fd, lambda: comp.on_bytes(tty.read(timeout=0)))
        try:
            while not done.is_set():          # the key parser needs periodic flushes (esc disambiguation)
                try: await asyncio.wait_for(done.wait(), 0.2)
                except asyncio.TimeoutError: comp.flush_input()
        finally:
            loop.remove_reader(tty.fd)
            comp.stop()
    finally:
        tty.write('\x1b[?2004l\x1b[?1000;1006l\r\n')
        tty.restore()
        agent.close()

The agent that reaches the terminal is a real one over real folders, gated on writes:

In [ ]:
tempdir = tempfile.mkdtemp()
cli_agent, cli_host = mk_agent([tempdir], approve='ask', web=False)
cli_host.roots, sorted(WRITE_TOOLS & set(t.__name__ for t in cli_agent.tools))

(['/private/var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/tmpr4g2iv5i'],
 ['add_cell',
  'create_file',
  'create_skill',
  'edit_cell',
  'edit_file',
  'run_python'])

In [ ]:
test_eq(cli_agent.approvals.mode, 'ask')
test_eq(mk_agent([tempdir], approve='none')[0].approvals, None)
cli_agent.model.name

'ornith-9b'

One turn with no terminal at all, for a pipe or a script -- and the entry point itself, which
is `ramabana` on the command line.

In [ ]:
#| export
def ask_once(agent, prompt):
    "One turn with no terminal at all, for a pipe or a script. Returns the exit code."
    print(agent.ask(prompt))
    for p in agent.problems: print(f'! {p}', file=sys.stderr)
    agent.close()
    return 0 if agent.ready else 1

In [ ]:
#| export
@call_parse
def main(
    prompt: str = '',                    # run one turn and exit; omit for the interactive session
    root: str = '.',                     # folders the agent may touch, comma separated
    model: str = None,                   # the turn model; the routing default when omitted
    approve: str = 'ask',                # ask | auto | off | none (gate nothing at all)
    web: bool = True,                    # let the web tools reach the network through fossick
    cfg: str = None,                     # config dir, for skills, extensions and history
):
    "Ramabana in a terminal: a coding agent over the folders you name."
    roots = [r.strip() for r in str(root).split(',') if r.strip()]
    agent, host = mk_agent(roots, model=model, approve=approve, web=web,
                           cfg=Path(cfg).expanduser() if cfg else None)
    if agent.start() is None and not prompt:
        print(f'no model available: {agent.note}', file=sys.stderr)
    if prompt: return sys.exit(ask_once(agent, prompt))
    hint = f"{', '.join(host.roots)} · /help"
    try: asyncio.run(amain(agent, hint))
    except KeyboardInterrupt: pass

`ask_once` prints the answer and reports anything that went wrong on stderr, so
`ramabana -p 'what changed?' | less` behaves like a Unix program.

In [ ]:
piped, _ = fake_agent(replies=['nothing changed since the last commit.'])
test_eq(ask_once(piped, 'what changed?'), 0)
piped.note

nothing changed since the last commit.


'fake · local · 1k ctx · 14 tools'

The command line, as `--help` prints it:

In [ ]:
#| eval: false
!ramabana --help

usage: ramabana [-h] [--prompt PROMPT] [--root ROOT] [--model MODEL]
                [--approve APPROVE] [--no-web] [--cfg CFG]

Ramabana in a terminal: a coding agent over the folders you name.

options:
  -h, --help         show this help message and exit
  --prompt PROMPT    run one turn and exit; omit for the interactive session
                     (default: )
  --root ROOT        folders the agent may touch, comma separated (default: .)
  --model MODEL      the turn model; the routing default when omitted
  --approve APPROVE  ask | auto | off | none (gate nothing at all) (default:
                     ask)
  --no-web           let the web tools reach the network through fossick
                     (default: True)
  --cfg CFG          config dir, for skills, extensions and history


```
usage: ramabana [-h] [--prompt PROMPT] [--root ROOT] [--model MODEL]
                [--approve APPROVE] [--no-web] [--cfg CFG]

Ramabana in a terminal: a coding agent over the folders you name.

  --prompt   run one turn and exit; omit for the interactive session
  --root     folders the agent may touch, comma separated (default: .)
  --model    the turn model; the routing default when omitted
  --approve  ask | auto | off | none (gate nothing at all) (default: ask)
  --no-web   keep the web tools off the network
  --cfg      config dir, for skills, extensions and history
```

So a session over this repository, on a local model, with nothing gated:

```bash
ramabana --root . --model qwen-4b --approve none
```

And one turn for a pipe, with no terminal at all:

```bash
ramabana --prompt 'which files import fastllm?'
```


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()